# TransHLA — Phase 2: Process Data (ESM2 embeddings)

Tính **sequence embedding** và **contact map** bằng `facebook/esm2_t33_650M_UR50D`
cho toàn bộ peptide từ Phase 1, rồi lưu xuống đĩa dạng `memmap` fp16 để Phase 3 huấn luyện
mà không phải chạy lại ESM2.

**Cần GPU.** `Runtime → Change runtime type → T4 GPU` (A100/L4 nhanh hơn 3–5×).

---

### Hai thứ cần tính (theo bài báo, Hình 1B)

| đầu ra | kích thước | dùng cho |
|---|---|---|
| sequence embedding `E` | `1280 × L` (layer 33 của ESM2) | nhánh Transformer + nhánh CNN 1 |
| contact map `S` | `L × L` (đối xứng) | nhánh CNN 2 |

Peptide được **đệm ở cuối bằng token id = 1** cho tới độ dài cố định
(`L = 16` cho HLA-I, `L = 23` cho HLA-II — khớp với model card chính thức trên HuggingFace:
tối đa 14 aa + `<cls>` + `<eos>` = 16, và 21 + 2 = 23).

### Dung lượng — đọc kỹ trước khi chạy

Mỗi peptide HLA-I: `1280×16×2 B` (embedding) + `16×16×2 B` (contact) ≈ **41 KB** (fp16).

| dữ liệu | số mẫu | dung lượng |
|---|---|---|
| HLA-I đầy đủ | 918.884 | **≈ 38 GB** |
| HLA-II đầy đủ | 624.490 | **≈ 38 GB** |

Google Drive bản miễn phí chỉ 15 GB ⇒ **mặc định notebook ghi vào đĩa cục bộ của Colab**
(`/content/cache`, ~100 GB nhưng mất khi ngắt phiên). Có 3 lựa chọn:

1. `SUBSET_N = 150_000` (mặc định) → ~6 GB mỗi lớp, **lưu được lên Drive**, đủ để tái hiện xu hướng kết quả.
2. `SUBSET_N = None` + `CACHE_ON_DRIVE = False` → đầy đủ, nhưng phải chạy Phase 2 và Phase 3 **trong cùng một phiên Colab**.
3. `SUBSET_N = None` + Drive có ≥ 80 GB (Colab Pro / Drive trả phí) → đầy đủ và lưu lâu dài.

Notebook này **có thể chạy tiếp sau khi bị ngắt** (resume) — nó ghi lại số mẫu đã xử lý.

## 0. Đồng bộ code từ GitHub

Code được sửa trong **VS Code** trên máy, `git push` lên GitHub, ô dưới đây kéo bản mới nhất về Colab.
Repo công khai nên chạy được ngay, không cần token.

> Ô này **ghi đè** mọi sửa đổi phía Colab. Nguồn sự thật là GitHub.


In [ ]:
# === Đồng bộ code từ GitHub — chạy ô này đầu tiên, mỗi phiên Colab ===
REPO = "github.com/Tienanh209/TransHLA_repro.git"
BRANCH = "main"

import os, sys, pathlib, subprocess

def _sh(*a):
    return subprocess.run(a, capture_output=True, text=True)

token = ""
try:
    from google.colab import userdata
    token = userdata.get("GH_TOKEN") or ""
except Exception:
    pass

url = f"https://{token + '@' if token else ''}{REPO}"
WORK = pathlib.Path("/content/TransHLA_repro")

if (WORK / ".git").exists():
    _sh("git", "-C", str(WORK), "remote", "set-url", "origin", url)
    _sh("git", "-C", str(WORK), "fetch", "origin", BRANCH)
    _sh("git", "-C", str(WORK), "reset", "--hard", f"origin/{BRANCH}")
else:
    r = _sh("git", "clone", "--branch", BRANCH, url, str(WORK))
    if r.returncode:
        raise SystemExit((r.stderr.replace(token, "***") if token else r.stderr))

os.chdir(WORK)
sys.path.insert(0, str(WORK))
print("code tại:", WORK, "|", _sh("git", "-C", str(WORK), "log", "-1", "--oneline").stdout.strip())


## 0. Cấu hình

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

In [ ]:
!pip -q install fair-esm transformers pyarrow tqdm

In [ ]:
import pathlib, os

PROJ = pathlib.Path('/content/drive/MyDrive/s1156052/EpitopePrediction')
DATA = PROJ / 'data'

# ---- các tham số bạn có thể đổi -------------------------------------------
HLA_CLASS      = 'I'        # 'I' hoặc 'II'  — chạy notebook 2 lần, mỗi lần một lớp
SUBSET_N       = 150_000    # None = dùng toàn bộ dữ liệu
CACHE_ON_DRIVE = True       # False = ghi vào /content (nhanh hơn, mất khi hết phiên)
BATCH          = 256        # giảm xuống nếu OOM (T4: 128–256, A100: 512)
ESM_MODEL      = 'esm2_t33_650M_UR50D'
# ---------------------------------------------------------------------------

MAXLEN = 16 if HLA_CLASS == 'I' else 23      # đã gồm <cls> và <eos>
EMB_DIM = 1280

CACHE = (PROJ / 'cache') if CACHE_ON_DRIVE else pathlib.Path('/content/cache')
CACHE = CACHE / f'hla{HLA_CLASS}'
CACHE.mkdir(parents=True, exist_ok=True)

print(f'lớp HLA-{HLA_CLASS} | MAXLEN={MAXLEN} | cache -> {CACHE}')
!df -h /content /content/drive 2>/dev/null | head -5

## 1. Nạp dữ liệu Phase 1

In [ ]:
import pandas as pd, numpy as np

df = pd.read_parquet(DATA / f'hla{HLA_CLASS}_all.parquet')
print(f'{len(df):,} dòng')
display(df.groupby(['split','label']).size().unstack(fill_value=0))

if SUBSET_N is not None and len(df) > SUBSET_N:
    # lấy mẫu phân tầng theo (split, label) để giữ nguyên cấu trúc dữ liệu
    frac = SUBSET_N / len(df)
    df = (df.groupby(['split','label'], group_keys=False)
            .apply(lambda g: g.sample(max(1, int(len(g)*frac)), random_state=42))
            .reset_index(drop=True))
    print(f'\n-> lấy mẫu con: {len(df):,} dòng')
    display(df.groupby(['split','label']).size().unstack(fill_value=0))

df = df.reset_index(drop=True)
df.to_parquet(CACHE / 'index.parquet', index=False)   # thứ tự dòng = thứ tự trong memmap
N = len(df)

## 2. Nạp ESM2

Dùng `fair-esm` (thư viện gốc của bài báo) vì nó cho ra **contact map** sẵn qua
`return_contacts=True`. ESM2-650M chiếm ~2.6 GB VRAM ở fp16.

In [ ]:
import torch, esm, time

device = 'cuda' if torch.cuda.is_available() else 'cpu'
assert device == 'cuda', 'Hãy bật GPU: Runtime -> Change runtime type -> T4 GPU'

model, alphabet = esm.pretrained.__dict__[ESM_MODEL]()
batch_converter = alphabet.get_batch_converter()
model = model.to(device).eval().half()        # fp16 cho nhanh
REPR_LAYER = model.num_layers                 # 33
print(f'{ESM_MODEL}: {sum(p.numel() for p in model.parameters())/1e6:.0f}M tham số, '
      f'layer biểu diễn = {REPR_LAYER}, pad_idx = {alphabet.padding_idx}')

In [ ]:
def encode_batch(seqs, maxlen=MAXLEN):
    # Token hoá rồi ĐỆM Ở CUỐI bằng token id = 1 (theo bài báo và model card).
    data = [(str(i), s) for i, s in enumerate(seqs)]
    _, _, toks = batch_converter(data)            # (B, L) có <cls> ... <eos>
    B, L = toks.shape
    if L < maxlen:
        pad = torch.full((B, maxlen - L), 1, dtype=toks.dtype)
        toks = torch.cat([toks, pad], dim=1)
    elif L > maxlen:
        toks = toks[:, :maxlen]
    return toks

# kiểm tra nhanh
print(encode_batch(['EDSAIVTPSR', 'SVWEPAKAKYVFR']))

## 3. Tạo memmap và chạy embedding (có thể resume)

Ba file được ghi ra:

* `emb.f16`    — `(N, MAXLEN, 1280)` sequence embedding
* `contact.f16`— `(N, MAXLEN, MAXLEN)` contact map
* `done.txt`   — số mẫu đã xử lý, để chạy tiếp nếu phiên bị ngắt

In [ ]:
emb_path = CACHE / 'emb.f16'
con_path = CACHE / 'contact.f16'
done_path = CACHE / 'done.txt'

emb_mm = np.memmap(emb_path, dtype=np.float16, mode='r+' if emb_path.exists() else 'w+',
                   shape=(N, MAXLEN, EMB_DIM))
con_mm = np.memmap(con_path, dtype=np.float16, mode='r+' if con_path.exists() else 'w+',
                   shape=(N, MAXLEN, MAXLEN))

start = int(done_path.read_text()) if done_path.exists() else 0
gb = (emb_mm.nbytes + con_mm.nbytes) / 1e9
print(f'memmap: {gb:.1f} GB  |  bắt đầu từ mẫu {start:,}/{N:,}')

In [ ]:
from tqdm.auto import tqdm

seqs = df['peptide'].tolist()
t0 = time.time()

with torch.no_grad():
    pbar = tqdm(range(start, N, BATCH), initial=start//BATCH, total=(N+BATCH-1)//BATCH)
    for i in pbar:
        chunk = seqs[i:i+BATCH]
        toks = encode_batch(chunk).to(device)
        out = model(toks, repr_layers=[REPR_LAYER], return_contacts=True)

        rep = out['representations'][REPR_LAYER]          # (B, MAXLEN, 1280)
        con = out['contacts']                             # (B, MAXLEN-2, MAXLEN-2)

        # contacts bỏ <cls>/<eos>; đệm lại về (MAXLEN, MAXLEN) để lưu đồng nhất
        B, Lc, _ = con.shape
        con_full = torch.zeros(B, MAXLEN, MAXLEN, device=con.device, dtype=con.dtype)
        con_full[:, 1:1+Lc, 1:1+Lc] = con

        emb_mm[i:i+len(chunk)] = rep.detach().cpu().numpy().astype(np.float16)
        con_mm[i:i+len(chunk)] = con_full.detach().cpu().numpy().astype(np.float16)

        if (i // BATCH) % 50 == 0:
            emb_mm.flush(); con_mm.flush()
            done_path.write_text(str(i + len(chunk)))
            rate = (i + len(chunk) - start) / max(1e-9, time.time() - t0)
            pbar.set_postfix(pep_per_s=f'{rate:.0f}', eta_min=f'{(N-i)/max(rate,1)/60:.0f}')

emb_mm.flush(); con_mm.flush(); done_path.write_text(str(N))
print(f'\nHoàn tất {N:,} peptide trong {(time.time()-t0)/60:.1f} phút')

## 4. Kiểm tra kết quả

In [ ]:
emb_r = np.memmap(emb_path, dtype=np.float16, mode='r', shape=(N, MAXLEN, EMB_DIM))
con_r = np.memmap(con_path, dtype=np.float16, mode='r', shape=(N, MAXLEN, MAXLEN))

print('embedding:', emb_r.shape, '| contact:', con_r.shape)
print('có NaN không? ', bool(np.isnan(emb_r[:1000].astype(np.float32)).any()))
print('biên độ embedding:', float(emb_r[:1000].astype(np.float32).min()),
      float(emb_r[:1000].astype(np.float32).max()))

import matplotlib.pyplot as plt
k = 0
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
ax[0].imshow(emb_r[k].astype(np.float32).T, aspect='auto', cmap='RdBu_r')
ax[0].set_title(f'embedding: {df.peptide[k]}'); ax[0].set_xlabel('vị trí'); ax[0].set_ylabel('chiều')
im = ax[1].imshow(con_r[k].astype(np.float32), cmap='viridis')
ax[1].set_title('contact map'); plt.colorbar(im, ax=ax[1])
plt.tight_layout(); plt.show()

In [ ]:
# Kiểm tra tính đối xứng của contact map (phải đối xứng theo định nghĩa)
c = con_r[k].astype(np.float32)
print('sai lệch đối xứng tối đa:', float(np.abs(c - c.T).max()))

# Ghi lại thông tin cấu hình để Phase 3 nạp đúng
import json
(CACHE / 'meta.json').write_text(json.dumps({
    'hla_class': HLA_CLASS, 'n': int(N), 'maxlen': MAXLEN, 'emb_dim': EMB_DIM,
    'esm_model': ESM_MODEL, 'subset_n': SUBSET_N, 'cache_on_drive': CACHE_ON_DRIVE,
}, indent=2))
print(open(CACHE / 'meta.json').read())

## 5. Chạy lại cho lớp HLA còn lại

Đổi `HLA_CLASS = 'II'` ở ô cấu hình (mục 0) rồi `Runtime → Restart and run all`.

---

### Thời gian tham khảo (ESM2-650M, fp16, batch 256)

| GPU | tốc độ | 150k peptide | toàn bộ HLA-I (919k) |
|---|---|---|---|
| T4 (Colab free) | ~600–800 pep/s | ~4 phút | **~25–35 phút** |
| L4 | ~1.800 pep/s | ~1,5 phút | ~9 phút |
| A100 | ~3.500 pep/s | <1 phút | ~5 phút |

Phần tính toán khá nhanh vì peptide rất ngắn; **nút cổ chai thực tế là ghi đĩa** —
nếu ghi thẳng lên Google Drive có thể chậm hơn 2–3 lần so với số trên.
Khi đó nên đặt `CACHE_ON_DRIVE = False`.

➡️ Tiếp theo: `03_run_experiment.ipynb`